In [1]:
import os

obj_storage_access_key = os.getenv('OBJ_STORAGE_ACCESS_KEY', 'qkN0JErJsKNNP5N8FAIx')
obj_storage_secret_key = os.getenv('OBJ_STORAGE_SECRET_KEY', 'n4tsviQPwHShkGLgzizAejqApfjdGjdjrmtUc8Sv')
obj_storage_endpoint = os.getenv('OBJ_STORAGE_ENDPOINT', 'e0g2.va01.idrivee2-76.com')

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

total_excuters = 2
executor_memory = 10
cores = 2
max_core = total_excuters * cores
parallelism =  int(max_core * 3)
shuffle = parallelism if parallelism > 180 else parallelism * 2
heap_memory = int(executor_memory * 0.15)
overhead_memory = executor_memory - heap_memory
connection = 5 * cores
print(f"""
total_excuters: {total_excuters},
executor_memory: {executor_memory},
cores: {cores},
max_core: {max_core}
parallelism: {parallelism}, 
shuffle: {shuffle},
heap_memory: {heap_memory}, 
overhead_memory: {overhead_memory},
connection: {connection}
""")

try:
    spark.catalog.clearCache()
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("test notebook2")
    .master("spark://spark-master:7077")
        
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3,io.delta:delta-spark_2.12:3.1.0,org.elasticsearch:elasticsearch-spark-30_2.12:8.12.2,org.apache.hadoop:hadoop-aws:3.3.1") 
    .config("spark.hadoop.fs.s3a.endpoint", obj_storage_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", obj_storage_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", obj_storage_secret_key)
    
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.databricks.delta.optimizeWrite.enabled", "false")
    .config("spark.databricks.delta.properties.defaults.enableChangeDataFeed", "true")
    .config("spark.databricks.delta.autoCompact.enabled", "true")
    
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "array")
        
    # --- Driver ---
    .config("spark.driver.memory", "24g")
        
    # --- Executor ---
    .config("spark.executor.instances", total_excuters)
    .config("spark.executor.cores", cores)
    .config("spark.cores.max", max_core)
    .config("spark.executor.memory", f"{executor_memory}g")
    .config("spark.executor.memoryOverhead", f"{overhead_memory}g")

    .config("spark.memory.fraction", "0.7")
    .config("spark.memory.storageFraction", "0.2")
        
    # --- Parallelism / Shuffle ---
    .config("spark.default.parallelism", parallelism)
    .config("spark.sql.shuffle.partitions", int(shuffle))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128MB")
    .config("spark.sql.adaptive.skewJoin.enabled", True)  # xử lý skewed keys
    .config("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
    .config("spark.sql.debugkers: 2kers: 2.maxToStringFields", 100)
    .config("spark.sql.files.maxPartitionBytes", "134217728")
    # --- S3A stability config ---
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000")
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000")
    .config("spark.hadoop.fs.s3a.threads.max", "32")
    .config("spark.hadoop.fs.s3a.readahead.range", "16M")
    .config("spark.hadoop.fs.s3a.attempts.maximum", "20")
    .config("spark.hadoop.fs.s3a.retry.limit", "10")
    .config("spark.hadoop.fs.s3a.retry.interval", "500ms")
    .config("spark.hadoop.fs.s3a.retry.throttle.limit", "60")
    .config("spark.hadoop.fs.s3a.paging.maximum", "100")
    .config("spark.hadoop.fs.s3a.experimental.input.fadvise", "sequential")
    .config("spark.hadoop.fs.s3a.multipart.size", "128M")
    .config("spark.speculation", "false")
    .config("spark.speculation.quantile", "0.8")
    .config("spark.speculation.multiplier", "1.5")
        
        
    .config("spark.pyspark.python", "python3.11")
    .config("spark.pyspark.driver.python", "python3.11")

    .config("spark.sql.autoBroadcastJoinThreshold", 50*1024*1024)  # 50MB
        
    .config("spark.sql.debug.maxToStringFields", 100)
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY")

    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "/opt/bitnami/spark/logs")
    .config("spark.history.fs.logDirectory", "/opt/bitnami/spark/logs")

    # .config("spark.sql.files.ignoreCorruptFiles", "true")
    # .config("spark.sql.files.ignoreMissingFiles", "true")
    # .config("spark.sql.json.schema.inferPartial", "true")

    .getOrCreate()
)


total_excuters: 2,
executor_memory: 10,
cores: 2,
max_core: 4
parallelism: 12, 
shuffle: 24,
heap_memory: 1, 
overhead_memory: 9,
connection: 10

:: loading settings :: url = jar:file:/opt/bitnami/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.postgresql#postgresql added as a dependency
io.delta#delta-spark_2.12 added as a dependency
org.elasticsearch#elasticsearch-spark-30_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7e922ef4-1557-4bd2-ad3c-aaa076d5beed;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.elasticsearch#elasticsearch-spark-30_2.12;8.12.2 in central
	found org.scala-lang#scala-reflect;2.12.8 in central
	found org.slf4j#slf4j-api;1.7.6 in central
	found commons-logging#commons-logging;1.1.1 in central
	found javax.xml.bind#jaxb-api;2.3.1 in central
	found com.google.protobuf#protob

In [3]:
dataset_2025 = spark.sql("""
        SELECT *
        FROM delta.`s3a://lakehouse-gold/trade_service/transaction_norm`
        WHERE period BETWEEN '2025-11' AND '2025-12'
""")
dataset_2025.createOrReplaceTempView("dataset_2025")
# "s3a://lakehouse-gold/trade_service/transaction_norm"

In [4]:
dataset_2025.count()

14609868